# Deep Ensemble — Supplementary Validation

Reproduces Table and reliability numbers in Appendix: *Deep Ensemble Validation: Per-Class Metrics*.

**What this notebook does:**
1. Loads 5 pre-trained ensemble members from `results_ensemble/`
2. Runs deterministic inference on the test set
3. Computes $C_k$ reliability stats ($\rho_k < 0.3$ per grade)
4. Runs 200-iteration bootstrap for AUSC and Critical FNR @80%

**To skip inference** (predictions already saved): jump to *Section 3 — Load Predictions*.

## 1. Setup

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import gc
import numpy as np
import tensorflow as tf
from tensorflow import keras
from scipy.stats import pearsonr

from modules.uncertaintybis import (
    compute_all_uncertainties,
    build_deferral_policies,
    compute_ausc,
    bootstrap_ausc,
    bootstrap_selective_prediction,
    selective_prediction_eval,
)
from modules.data_pipeline import (
    load_and_combine_datasets,
    patient_level_split,
    make_dataset,
)
from modules.efficientnet_builder import build_efficientnet_dr

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)

# ── Configuration ─────────────────────────────────────────────────────────────
N_MEMBERS        = 5
IMG_SIZE         = 256
BATCH_SIZE       = 32
SEED             = 42
CRITICAL_CLASSES = [2, 3]   # per paper
SAFE_CLASSES     = [0, 1]
RELIABILITY_THR  = 0.3      # rho_k threshold
N_BOOTSTRAP      = 200
COVERAGE_80      = 0.80
ENSEMBLE_DIR     = 'results_ensemble'

print(f'TensorFlow : {tf.__version__}')
print(f'GPUs       : {tf.config.list_physical_devices("GPU")}')
print(f'Critical   : {CRITICAL_CLASSES}  Safe: {SAFE_CLASSES}')

2026-02-22 13:24:45.596422: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-22 13:24:45.596458: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-22 13:24:45.597444: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow : 2.15.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Critical   : [2, 3]  Safe: [0, 1]


## 2. Inference  *(skip if `ensemble_preds.npy` already exists)*

## 3. Load Predictions

In [3]:
ensemble_preds = np.load(PREDS_PATH)   # [5, N_test, 4]
y_true         = np.load(YTRUE_PATH).astype(int)

print(f'Ensemble predictions : {ensemble_preds.shape}')
print(f'Test samples         : {len(y_true)}')
print(f'Class distribution   : {np.bincount(y_true)}')

# Compute all uncertainty metrics
metrics  = compute_all_uncertainties(ensemble_preds, CRITICAL_CLASSES, SAFE_CLASSES)
policies = build_deferral_policies(metrics)
y_pred   = metrics['y_pred']

overall_acc = np.mean(y_pred == y_true)
print(f'\nEnsemble accuracy    : {overall_acc:.4f}')

Ensemble predictions : (5, 7948, 4)
Test samples         : 7948
Class distribution   : [5584  587 1322  455]

Ensemble accuracy    : 0.8250


## 4. $C_k$ Reliability  
Fraction of samples satisfying $\rho_k < 0.3$ per grade.

In [4]:
rho_k      = metrics['rho_k']   # [N, 4]
grade_names = ['Grade 0 (No DR)', 'Grade 1 (Mild)', 'Grade 2 (Moderate)', 'Grade 3 (Severe/PDR)']

print('C_k Reliability (rho_k < 0.3):')
print(f'  {"Grade":<22} {"% reliable":>12} {"median rho_k":>14} {"n samples":>10}')
print('  ' + '-'*60)
for k, name in enumerate(grade_names):
    pct    = np.mean(rho_k[:, k] < RELIABILITY_THR) * 100
    med    = np.median(rho_k[:, k])
    n      = (y_true == k).sum()
    print(f'  {name:<22} {pct:>11.1f}% {med:>14.4f} {n:>10}')

# Taylor approximation quality: sum_C_k vs MI
r_pearson, _ = pearsonr(metrics['sum_C_k'], metrics['MI'])
print(f'\nTaylor approximation quality:')
print(f'  Pearson r(sum_k C_k, MI) = {r_pearson:.4f}')
print(f'  --> Paper reports: 0.996')

C_k Reliability (rho_k < 0.3):
  Grade                    % reliable   median rho_k  n samples
  ------------------------------------------------------------
  Grade 0 (No DR)               95.7%         0.0060       5584
  Grade 1 (Mild)                91.2%         0.0678        587
  Grade 2 (Moderate)            91.5%         0.0590       1322
  Grade 3 (Severe/PDR)          85.1%         0.0932        455

Taylor approximation quality:
  Pearson r(sum_k C_k, MI) = 0.9959
  --> Paper reports: 0.996


## 5. Bootstrap AUSC + Critical FNR @80%  
200 resamples. Policies: the 5 in Table~\ref{tab:deep-ensemble-sale} + MI reference.

In [5]:
TABLE_POLICIES = {
    'Var_crit'       : policies['Var_critical'],
    'Sale_EU_crit'   : policies['Sale_EU_critical'],
    'C_crit_max'     : policies['C_critical_max'],
    'C_crit_sum'     : policies['C_critical_sum'],
    'CBEC'           : policies['CBEC'],
    'MI'             : policies['MI'],     # reference (not in table)
}

# AUSC bootstrap
print('Running AUSC bootstrap...')
boot_ausc = bootstrap_ausc(
    y_true, y_pred, TABLE_POLICIES,
    critical_classes=CRITICAL_CLASSES,
    n_bootstrap=N_BOOTSTRAP, seed=SEED, metric='critical_fnr',
)

# @80% bootstrap
print('Running @80% bootstrap...')
boot_80 = bootstrap_selective_prediction(
    y_true, y_pred, TABLE_POLICIES,
    coverage=COVERAGE_80,
    critical_classes=CRITICAL_CLASSES,
    n_bootstrap=N_BOOTSTRAP, seed=SEED,
)
print('Done.')

Running AUSC bootstrap...
Running bootstrap with 200 iterations...
  Bootstrap 200/200
  Bootstrap complete!
Running @80% bootstrap...
Running bootstrap at coverage=0.80 with 200 iterations...
  Bootstrap 200/200
  Bootstrap complete!
Done.


In [6]:
TABLE_ORDER = ['Var_crit', 'Sale_EU_crit', 'C_crit_max', 'C_crit_sum', 'CBEC']

print('=' * 70)
print('TABLE: Deep Ensemble — Per-Class Metrics')
print(f'Critical classes: {CRITICAL_CLASSES}  |  Bootstrap: {N_BOOTSTRAP} iterations')
print('=' * 70)
print(f'  {"Policy":<16} {"AUSC (mean±std)":>20} {"FNR@80% (mean±std)":>22}')
print('  ' + '-' * 62)
for pol in TABLE_ORDER:
    s_ausc = boot_ausc['summary'][pol]
    s_fnr  = boot_80[pol]['critical_fnr']
    marker = '**' if pol == 'CBEC' else '  '
    print(f'{marker} {pol:<16} '
          f'{s_ausc["mean"]:>8.3f} +/- {s_ausc["std"]:>5.3f}   '
          f'{s_fnr["mean"]:>8.3f} +/- {s_fnr["std"]:>5.3f}')
print('=' * 70)

# MI reference for text
mi_ausc  = boot_ausc['summary']['MI']['mean']
cbec_ausc = boot_ausc['summary']['CBEC']['mean']
sale_ausc = boot_ausc['summary']['Sale_EU_crit']['mean']
ck_ausc   = boot_ausc['summary']['C_crit_max']['mean']

print(f'\nKey numbers:')
print(f'  MI AUSC                       : {mi_ausc:.3f}')
print(f'  CBEC vs MI improvement        : {(mi_ausc - cbec_ausc)/mi_ausc*100:.1f}%  ({cbec_ausc:.3f} vs {mi_ausc:.3f})')
print(f'  C_crit_max vs Sale_EU_crit    : {(sale_ausc - ck_ausc)/sale_ausc*100:.1f}%  ({ck_ausc:.3f} vs {sale_ausc:.3f})')
print(f'  Taylor r(sum_k C_k, MI)       : {r_pearson:.3f}')

print(f'\nReliability numbers for paper text:')
for k in range(4):
    pct = np.mean(rho_k[:, k] < RELIABILITY_THR) * 100
    print(f'  Grade {k}: {pct:.1f}%')

TABLE: Deep Ensemble — Per-Class Metrics
Critical classes: [2, 3]  |  Bootstrap: 200 iterations
  Policy                AUSC (mean±std)     FNR@80% (mean±std)
  --------------------------------------------------------------
   Var_crit            0.408 +/- 0.029      0.364 +/- 0.017
   Sale_EU_crit        0.447 +/- 0.031      0.414 +/- 0.018
   C_crit_max          0.390 +/- 0.029      0.314 +/- 0.016
   C_crit_sum          0.406 +/- 0.029      0.333 +/- 0.016
** CBEC                0.223 +/- 0.018      0.237 +/- 0.013

Key numbers:
  MI AUSC                       : 0.354
  CBEC vs MI improvement        : 36.9%  (0.223 vs 0.354)
  C_crit_max vs Sale_EU_crit    : 12.8%  (0.390 vs 0.447)
  Taylor r(sum_k C_k, MI)       : 0.996

Reliability numbers for paper text:
  Grade 0: 95.7%
  Grade 1: 91.2%
  Grade 2: 91.5%
  Grade 3: 85.1%
